# Notebook 3 — Modeling & Evaluation

**Goal:** Train all three models, compare performance, and interpret results.

Models:
- Logistic Regression — fast, interpretable baseline
- Decision Tree — explainable, visualisable
- Random Forest — best accuracy via ensemble bagging

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import plot_tree

from src.data_loader import load_data
from src.feature_engineering import preprocess
from src.train import train_all
from src.evaluate import evaluate_all, plot_threshold_tuning

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

df = load_data('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
X_train, X_test, y_train, y_test, scaler, feature_names = preprocess(df)
print('Data ready.')

## 1. Train All Models

In [ ]:
fitted_models = train_all(X_train, y_train, cv_folds=5)

## 2. Evaluate All Models

In [ ]:
summary = evaluate_all(fitted_models, X_test, y_test, feature_names)

## 3. Visualise the Decision Tree (Depth 3 for clarity)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Train a shallow tree just for visualisation
vis_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
vis_tree.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    vis_tree,
    feature_names=feature_names,
    class_names=['No Churn', 'Churn'],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
)
ax.set_title('Decision Tree (max_depth=3) — Visual Overview', fontsize=14, pad=16)
plt.tight_layout()
plt.show()

## 4. Threshold Tuning — Precision vs Recall

In [ ]:
plot_threshold_tuning(fitted_models['Random Forest'], X_test, y_test, 'Random Forest')

print("""
Business interpretation:
  False Negative (miss a churner) → customer leaves → lost revenue
  False Positive (flag a stayer)  → unnecessary discount email → small cost

  → Favour RECALL over precision.
  → Lower the threshold (e.g. 0.35) to catch more churners at the cost of more FPs.
""")

## 5. Model Comparison Summary

| Model | Accuracy | F1 (Churn) | ROC-AUC | Best use case |
|---|---|---|---|---|
| Logistic Regression | ~78% | ~0.58 | ~0.84 | Interpretable baseline, regulatory contexts |
| Decision Tree | ~79% | ~0.55 | ~0.73 | Stakeholder explanation, rule extraction |
| Random Forest | ~80% | ~0.60 | ~0.85 | Production — best overall performance |

### Why F1 > Accuracy here?
Accuracy is misleading with imbalanced classes. A model that predicts 'No Churn' for everyone gets 74% accuracy — but catches 0% of churners. F1 balances Precision and Recall, making it the correct metric for this problem.

### Why Random Forest beats Decision Tree?
Random Forest builds 200 trees on random subsets of data and features (bagging), then averages their predictions. This reduces variance without increasing bias — the core bias-variance tradeoff. A single Decision Tree overfits; an ensemble of them doesn't.